# vLLM Tool Calling Demo (Offline / Notebook Style)

In offline mode (`vllm.LLM`), the output is raw text — `enable_auto_tool_choice` and `tool_call_parser` are server-only flags. But you **can** use vLLM's built-in `ToolParser` classes standalone to parse the output.

The workflow is:
1. Pass `tools` to `llm.chat()` — the model's chat template formats them into the prompt
2. Model generates text with `<tool_call>` tags
3. Use `Hermes2ProToolParser.extract_tool_calls()` to get structured tool calls

## 1. Setup

In [12]:
import json

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from vllm.tool_parsers.hermes_tool_parser import Hermes2ProToolParser

MODEL = "Qwen/Qwen3-4B"  # Any model whose chat template supports tools

llm = LLM(model=MODEL, dtype="bfloat16")
sampling_params = SamplingParams(max_tokens=2048, temperature=0.0)

# Set up the tool call parser — it just needs a tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
parser = Hermes2ProToolParser(tokenizer)

INFO 04-09 00:44:24 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B'}
INFO 04-09 00:44:24 [model.py:533] Resolved architecture: Qwen3ForCausalLM
INFO 04-09 00:44:24 [model.py:1582] Using max model len 40960
INFO 04-09 00:44:24 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=16384.


Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


(EngineCore pid=2783460) INFO 04-09 00:44:32 [core.py:103] Initializing a V1 LLM engine (v0.18.0) with config: model='Qwen/Qwen3-4B', speculative_config=None, tokenizer='Qwen/Qwen3-4B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None,

(EngineCore pid=2783460) <frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=2783460) <frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:01,  1.61it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:01<00:00,  1.47it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.09it/s]
(EngineCore pid=2783460) 


(EngineCore pid=2783460) INFO 04-09 00:44:36 [default_loader.py:384] Loading weights took 1.44 seconds
(EngineCore pid=2783460) INFO 04-09 00:44:37 [gpu_model_runner.py:4566] Model loading took 7.56 GiB memory and 2.177249 seconds
(EngineCore pid=2783460) INFO 04-09 00:44:40 [backends.py:988] Using cache directory: /home/oumi/.cache/vllm/torch_compile_cache/175083d880/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=2783460) INFO 04-09 00:44:40 [backends.py:1048] Dynamo bytecode transform time: 2.72 s
(EngineCore pid=2783460) INFO 04-09 00:44:41 [backends.py:284] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 1.267 s
(EngineCore pid=2783460) INFO 04-09 00:44:41 [monitor.py:48] torch.compile took 4.61 s in total
(EngineCore pid=2783460) INFO 04-09 00:44:41 [decorators.py:296] Directly load AOT compilation from path /home/oumi/.cache/vllm/torch_compile_cache/torch_aot_compile/c27b42f2fb9dae0b4885276e68e72b343fe1fcb52038e374e9ecc195bbee351c

(EngineCore pid=2783460) 2026-04-09 00:44:43,997 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=2783460) 2026-04-09 00:44:44,010 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 21.55it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:01<00:00, 34.68it/s]


(EngineCore pid=2783460) INFO 04-09 00:44:48 [gpu_model_runner.py:5746] Graph capturing finished in 4 secs, took 0.96 GiB
(EngineCore pid=2783460) INFO 04-09 00:44:48 [gpu_worker.py:617] CUDA graph pool memory: 0.96 GiB (actual), 0.74 GiB (estimated), difference: 0.21 GiB (22.1%).
(EngineCore pid=2783460) INFO 04-09 00:44:48 [core.py:281] init engine (profile, create kv cache, warmup model) took 11.32 seconds
(EngineCore pid=2783460) INFO 04-09 00:44:49 [vllm.py:754] Asynchronous scheduling is enabled.
INFO 04-09 00:44:49 [llm.py:391] Supported tasks: ['generate']


## 2. Define tools

Tool definitions follow the OpenAI function-calling schema.

In [13]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City and state, e.g. 'San Francisco, CA'",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit",
                    },
                },
                "required": ["location"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query",
                    },
                    "num_results": {
                        "type": "integer",
                        "description": "Number of results to return",
                    },
                },
                "required": ["query"],
            },
        },
    },
]

print(f"Defined {len(tools)} tools: {[t['function']['name'] for t in tools]}")

Defined 2 tools: ['get_weather', 'web_search']


## 3. Single-turn tool call

Pass `tools` to `llm.chat()`. The model's chat template (Qwen3's Jinja2 template) injects the tool definitions into the prompt. The model outputs `<tool_call>` tags which we parse.

In [14]:
messages = [
    {"role": "user", "content": "What's the weather like in Tokyo right now?"},
]

outputs = llm.chat(
    messages=messages,
    sampling_params=sampling_params,
    tools=tools,  # Chat template formats these into the prompt
)

raw_text = outputs[0].outputs[0].text
print("=== Raw model output ===")
print(raw_text)
print()

# Use vLLM's built-in Hermes parser (same one used by vllm serve)
result = parser.extract_tool_calls(raw_text, request=None)
if result.tools_called:
    print("=== Parsed tool calls ===")
    for tc in result.tool_calls:
        print(f"  Function: {tc.function.name}")
        print(f"  Arguments: {tc.function.arguments}")
    if result.content:
        print(f"  Content (before tool call): {result.content[:200]}")
else:
    print("No tool call detected — model responded with plain text.")

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

=== Raw model output ===
<think>
Okay, the user is asking about the weather in Tokyo. Let me check the tools available. There's a get_weather function that requires location and optional unit. Since they didn't specify Celsius or Fahrenheit, maybe I should default to one. The function requires location, so I'll use "Tokyo, Japan" as the location. The unit isn't specified, but maybe the function has a default. Let me make sure to include the location parameter. I'll call get_weather with location set to Tokyo, Japan. If the user wants a specific unit, they can clarify, but the function might handle the default based on their location. Alright, that's the plan.
</think>

<tool_call>
{"name": "get_weather", "arguments": {"location": "Tokyo, Japan", "unit": "celsius"}}
</tool_call>

=== Parsed tool calls ===
  Function: get_weather
  Arguments: {"location": "Tokyo, Japan", "unit": "celsius"}
  Content (before tool call): <think>
Okay, the user is asking about the weather in Tokyo. Let me c

## 4. Multi-turn: Feed tool result back

After parsing the tool call, simulate executing it and send the result back as a `tool` message. The model then generates a natural language answer.

In [15]:
# Simulate executing the tool
tool_result = json.dumps(
    {
        "temperature": 22,
        "unit": "celsius",
        "condition": "partly cloudy",
    }
)

# Build multi-turn conversation:
#   user -> assistant (with tool call text) -> tool (result) -> assistant (final answer)
multi_turn_messages = [
    {"role": "user", "content": "What's the weather like in Tokyo right now?"},
    # The assistant's turn — include the raw text with <tool_call> tags
    {"role": "assistant", "content": raw_text},
    # The tool result
    {"role": "tool", "content": tool_result},
]

outputs = llm.chat(
    messages=multi_turn_messages,
    sampling_params=sampling_params,
    tools=tools,
)

print("=== Final answer after tool result ===")
print(outputs[0].outputs[0].text)

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

=== Final answer after tool result ===
<think>
Okay, the user asked about the weather in Tokyo. I called the get_weather function with location set to "Tokyo, Japan" and unit as celsius. The response came back with temperature 22°C and partly cloudy. Now I need to present this information clearly. Let me check if the unit was specified in the query. The user didn't mention a unit, so the function's default probably used celsius. I should state the temperature and condition clearly. Maybe add a friendly note about the weather being pleasant. Make sure the answer is concise and addresses the user's question directly.
</think>

The current weather in Tokyo, Japan is **22°C** with **partly cloudy** conditions. It's a mild day—perfect for outdoor activities! 🌤️


---
## 5. Batch tool calling

Pass a list of message lists for parallel inference.

In [16]:
batch_messages = [
    [{"role": "user", "content": "What's the weather in Tokyo?"}],
    [{"role": "user", "content": "Search for the latest vLLM release notes."}],
    [{"role": "user", "content": "What's the weather in London in fahrenheit?"}],
]

outputs = llm.chat(
    messages=batch_messages,
    sampling_params=sampling_params,
    tools=tools,
)

for i, output in enumerate(outputs):
    raw = output.outputs[0].text
    result = parser.extract_tool_calls(raw, request=None)
    print(f"--- Request {i} ---")
    if result.tools_called:
        for tc in result.tool_calls:
            print(f"  Tool: {tc.function.name}({tc.function.arguments})")
    else:
        print(f"  Text: {raw[:200]}")
    print()

Rendering conversations:   0%|          | 0/3 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

--- Request 0 ---
  Tool: get_weather({"location": "Tokyo, Japan", "unit": "celsius"})

--- Request 1 ---
  Tool: web_search({"query": "latest vLLM release notes", "num_results": 5})

--- Request 2 ---
  Tool: get_weather({"location": "London", "unit": "fahrenheit"})



---
## Notes

- **Offline vs Server**: `enable_auto_tool_choice` and `tool_call_parser` are init flags for `vllm serve` only — not available on the `LLM` class. But you can import the same parser classes (`Hermes2ProToolParser`, etc.) and use them standalone.
- **Available parsers**: `from vllm.tool_parsers import ToolParserManager; ToolParserManager.list_registered()` — includes `hermes`, `llama3_json`, `mistral`, `qwen3_coder`, `qwen3_xml`, `deepseek_v3`, and many more.
- **Chat template**: Qwen3's built-in template handles the `tools` kwarg natively (formats them into the system prompt and expects `<tool_call>` output). Other models (Llama 3.1+, Mistral) have their own formats and parsers.
- **Qwen3 thinking**: Qwen3 may include `<think>...</think>` blocks before the tool call. The Hermes parser handles this — `result.content` contains everything before the first `<tool_call>` tag.